In [2]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [3]:
project_root = os.path.abspath("../..")

models_dir = os.path.join(
    project_root,
    "ml-model",
    "models"
)

preprocessing_dir = os.path.join(
    project_root,
    "preprocessing"
)

if models_dir not in sys.path:
    sys.path.insert(0, models_dir)

if preprocessing_dir not in sys.path:
    sys.path.insert(0, preprocessing_dir)

In [4]:
import resnet50_model
import importlib
importlib.reload(resnet50_model)


<module 'resnet50_model' from 'd:\\AI-Powered-Diabetic-Retinopathy-System\\ml-model\\models\\resnet50_model.py'>

In [5]:
model = resnet50_model.build_resnet50_model(
    trainable=False
)

print("ResNet50 model created!")

ResNet50 model created!


In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("ResNet50 compiled successfully!")

ResNet50 compiled successfully!


In [7]:
import os
import sys
import importlib

import tensorflow as tf
import pandas as pd
import numpy as np

project_root = os.path.abspath("..")

preprocessing_dir = os.path.join(
    project_root,
    "preprocessing"
)

models_dir = os.path.join(
    project_root,
    "models"
)

config_dir = os.path.join(
    project_root,
    "config"
)

for directory in [
    preprocessing_dir,
    models_dir,
    config_dir
]:
    if directory not in sys.path:
        sys.path.insert(0, directory)

import dataset_loader
import class_weights
import efficientnet_model
import train_model

importlib.reload(dataset_loader)
importlib.reload(class_weights)
importlib.reload(efficientnet_model)
importlib.reload(train_model)

print("All modules imported successfully!")

All modules imported successfully!


In [8]:
import dataset_loader
importlib.reload(dataset_loader)

<module 'dataset_loader' from 'd:\\AI-Powered-Diabetic-Retinopathy-System\\ml-model\\preprocessing\\dataset_loader.py'>

In [9]:
csv_path = "D:\\AI-Powered-Diabetic-Retinopathy-System\\ml-model\\dataset\\raw\\train.csv"

df = dataset_loader.load_metadata(csv_path)

image_dir = "D:\\AI-Powered-Diabetic-Retinopathy-System\\ml-model\\dataset\\raw\\train_images"

df = dataset_loader.create_image_paths(
    df,
    image_dir
)

df = dataset_loader.validate_image_paths(df)

print("Total images:", len(df))

Total images: 3662


In [10]:
train_df, validation_df, test_df = (
    dataset_loader.split_dataset(df)
)

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Training: 2563
Validation: 549
Test: 550


In [11]:
weights = class_weights.calculate_class_weights(
    train_df["diagnosis"].values
)

In [12]:


importlib.reload(dataset_loader)
train_dataset_resnet = dataset_loader.create_tf_dataset(
    train_df,
    batch_size=8,
    shuffle=True,
    training=True
)

validation_dataset_resnet = dataset_loader.create_tf_dataset(
    validation_df,
    batch_size=8,
    shuffle=False,
    training=False
)

test_dataset_resnet = dataset_loader.create_tf_dataset(
    test_df,
    batch_size=8,
    shuffle=False,
    training=False
)

print("ResNet50 datasets created successfully!")

ResNet50 datasets created successfully!


In [13]:
for images, labels in train_dataset_resnet.take(1):

    print("Images:", images.shape)
    print("Labels:", labels.shape)

    predictions = model.predict(
        images,
        verbose=0
    )

    print("Predictions:", predictions.shape)

Images: (8, 224, 224, 3)
Labels: (8,)
Predictions: (8, 5)


In [14]:
import callbacks
importlib.reload(callbacks)

<module 'callbacks' from 'd:\\AI-Powered-Diabetic-Retinopathy-System\\ml-model\\models\\callbacks.py'>

In [ ]:
_finetune_callbacks = callbacks.create_training_callbacks(
    "../saved_models/best_resnet50.keras"
)

In [16]:
resnet_history = model.fit(
    train_dataset_resnet,
    validation_data=validation_dataset_resnet,
    epochs=5,
    class_weight=weights,
    callbacks=finetune_callbacks
)

Epoch 1/5


d:\AI-Powered-Diabetic-Retinopathy-System\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


321/321 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step - accuracy: 0.5759 - loss: 1.3839
Epoch 1: val_loss improved from None to 0.82252, saving model to ../saved_models/best_resnet50.keras

Epoch 1: finished saving model to ../saved_models/best_resnet50.keras
321/321 ━━━━━━━━━━━━━━━━━━━━ 153s 458ms/step - accuracy: 0.5759 - loss: 1.3839 - val_accuracy: 0.6958 - val_loss: 0.8225 - learning_rate: 0.0010
Epoch 2/5
321/321 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step - accuracy: 0.6660 - loss: 1.1497
Epoch 2: val_loss did not improve from 0.82252
321/321 ━━━━━━━━━━━━━━━━━━━━ 201s 454ms/step - accuracy: 0.6660 - loss: 1.1497 - val_accuracy: 0.6011 - val_loss: 1.0680 - learning_rate: 0.0010
Epoch 3/5
321/321 ━━━━━━━━━━━━━━━━━━━━ 0s 538ms/step - accuracy: 0.6785 - loss: 1.1015
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.

Epoch 3: val_loss did not improve from 0.82252
321/321 ━━━━━━━━━━━━━━━━━━━━ 200s 622ms/step - accuracy: 0.6785 - loss: 1.1015 - val_accuracy: 0.6248 - val_loss: 1

In [19]:
best_epoch = np.argmin(
    resnet_history.history["val_loss"]
) + 1

best_val_loss = min(
    resnet_history.history["val_loss"]
)

best_val_accuracy = max(
    resnet_history.history["val_accuracy"]
)

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Best validation accuracy:", best_val_accuracy)

Best epoch: 5
Best validation loss: 0.7141528725624084
Best validation accuracy: 0.7213114500045776


In [20]:
best_resnet_model = tf.keras.models.load_model(
    "../saved_models/best_resnet50.keras"
)

print("Best ResNet50 model loaded successfully!")

Best ResNet50 model loaded successfully!


In [21]:
test_loss, test_accuracy = best_resnet_model.evaluate(
    test_dataset_resnet,
    verbose=1
)

print("\nResNet50 Test Results")
print("------------------------------")
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

69/69 ━━━━━━━━━━━━━━━━━━━━ 27s 348ms/step - accuracy: 0.7564 - loss: 0.6287

ResNet50 Test Results
------------------------------
Test Loss: 0.6286581158638
Test Accuracy: 0.7563636302947998


In [22]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Get predictions
y_true = []
y_pred = []

for images, labels in test_dataset_resnet:
    predictions = best_resnet_model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Class names
class_names = [
    "No DR",
    "Mild DR",
    "Moderate DR",
    "Severe DR",
    "Proliferative DR"
]

# Classification report
print("ResNet50 Classification Report")
print("--------------------------------")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("\nResNet50 Confusion Matrix")
print("--------------------------------")
print(cm)

ResNet50 Classification Report
--------------------------------
                  precision    recall  f1-score   support

           No DR     0.9527    0.9668    0.9597       271
         Mild DR     0.4737    0.6429    0.5455        56
     Moderate DR     0.7456    0.5667    0.6439       150
       Severe DR     0.4286    0.3103    0.3600        29
Proliferative DR     0.3750    0.5455    0.4444        44

        accuracy                         0.7564       550
       macro avg     0.5951    0.6064    0.5907       550
    weighted avg     0.7736    0.7564    0.7586       550


ResNet50 Confusion Matrix
--------------------------------
[[262   7   2   0   0]
 [  5  36  11   0   4]
 [  7  29  85   8  21]
 [  0   0   5   9  15]
 [  1   4  11   4  24]]
